# Jarvis George — Remote Inference Server (Dual Model)

**Jarvis George Digital Twin — Diploma Thesis**

This notebook loads the fine-tuned model and exposes an
Ollama-compatible API via ngrok, so the n8n workflow on the Mac Mini
can call it remotely.

**Model strategy (fallback):**
1. **Primary:** Mistral-7B + Part A LoRA adapters (fine-tuned on Persona-Chat)
2. **Fallback:** Krikri-8B base (Greek-focused, no adapter)

**Steps:**
1. Install dependencies
2. Mount Google Drive + Configure ngrok
3. Load model (auto-selects primary or fallback)
4. Start FastAPI server (Ollama-compatible `/api/chat`)
5. Copy the ngrok URL to n8n's `JARVIS_OLLAMA_URL`

In [ ]:
# ── Step 1: Install dependencies ──
!pip install -q "transformers>=4.44" accelerate "bitsandbytes>=0.46.1" peft
!pip install -q fastapi uvicorn pyngrok nest_asyncio

In [ ]:
# ── Step 2: Mount Drive + Configure ngrok ──
from google.colab import drive
drive.mount("/content/drive")

# Get your free auth token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- ΒΑΛΕ ΤΟ ΔΙΚΟ ΣΟΥ

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
# ── Step 3: Load model (Primary: Mistral-7B + adapters, Fallback: Krikri-8B) ──
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ─── CONFIGURATION ───
# Primary: Mistral-7B with your fine-tuned Part A adapters
PRIMARY_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_PATH = "/content/drive/MyDrive/jarvis_models/persona_chat_qlora"

# Fallback: Krikri-8B (Greek-focused, no adapter needed)
FALLBACK_MODEL = "ilsp/Llama-Krikri-8B-Instruct"

# ─── AUTO-SELECT ───
adapter_exists = Path(ADAPTER_PATH).exists() and (Path(ADAPTER_PATH) / "adapter_model.safetensors").exists()

if adapter_exists:
    BASE_MODEL = PRIMARY_MODEL
    USE_ADAPTER = True
    MODEL_LABEL = "mistral-7b-persona"
    print(f"✅ Part A adapters found at {ADAPTER_PATH}")
    print(f"→ Loading PRIMARY: {PRIMARY_MODEL} + LoRA adapter")
else:
    BASE_MODEL = FALLBACK_MODEL
    USE_ADAPTER = False
    MODEL_LABEL = "krikri-8b"
    print(f"⚠️ No adapters at {ADAPTER_PATH}")
    print(f"→ Loading FALLBACK: {FALLBACK_MODEL} (no adapter)")

# ─── LOAD ───
print(f"\nLoading tokenizer: {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model with 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

if USE_ADAPTER:
    print(f"Loading LoRA adapter from {ADAPTER_PATH}...")
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    print("✅ Adapter loaded!")

model.eval()
print(f"\n{'='*50}")
print(f"Model: {MODEL_LABEL}")
print(f"Device: {model.device}")
print(f"Memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"{'='*50}")

In [ ]:
# ── Step 4: Define Ollama-compatible FastAPI server ──
import time
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="Jarvis George Inference Server", version="2.0")


# ── Ollama-compatible request/response models ──

class ChatMessage(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    model: str = "jarvis"
    messages: list[ChatMessage]
    stream: bool = False
    options: dict = Field(default_factory=dict)


class ChatResponse(BaseModel):
    model: str
    created_at: str
    message: ChatMessage
    done: bool = True
    total_duration: int = 0
    eval_count: int = 0


def build_prompt(messages: list[ChatMessage]) -> str:
    """Build prompt in Mistral/Llama instruct format."""
    parts = []
    for msg in messages:
        if msg.role == "system":
            parts.append(f"[INST] <<SYS>>\n{msg.content}\n<</SYS>>")
        elif msg.role == "user":
            if parts and parts[-1].startswith("[INST] <<SYS>>"):
                parts[-1] += f"\n{msg.content} [/INST]"
            else:
                parts.append(f"[INST] {msg.content} [/INST]")
        elif msg.role == "assistant":
            parts.append(msg.content)
    return "\n".join(parts)


def generate_response(messages: list[ChatMessage], options: dict) -> tuple[str, int]:
    """Generate response using the loaded model."""
    prompt = build_prompt(messages)

    max_tokens = options.get("num_predict", 150)
    temperature = options.get("temperature", 0.5)
    top_p = options.get("top_p", 0.85)
    top_k_val = options.get("top_k", 40)
    repetition_penalty = options.get("repeat_penalty", 1.2)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=max(temperature, 0.01),
            top_p=top_p,
            top_k=top_k_val,
            repetition_penalty=repetition_penalty,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response_text, len(new_tokens)


# ── Endpoints ──

@app.post("/api/chat")
def chat(req: ChatRequest):
    start = time.time()
    reply, num_tokens = generate_response(req.messages, req.options)
    duration = int((time.time() - start) * 1e9)

    return ChatResponse(
        model=MODEL_LABEL,
        created_at=time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        message=ChatMessage(role="assistant", content=reply),
        done=True,
        total_duration=duration,
        eval_count=num_tokens,
    )


@app.get("/api/tags")
def list_models():
    return {
        "models": [{
            "name": MODEL_LABEL,
            "modified_at": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "size": 5_000_000_000,
        }]
    }


@app.get("/health")
def health():
    return {"status": "ok", "twin_enabled": True, "model": MODEL_LABEL}


print(f"FastAPI server defined — model: {MODEL_LABEL}")

In [ ]:
# ── Step 5: Start server + ngrok tunnel ──
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()

# Start ngrok tunnel
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url  # extract just the URL string
print("=" * 60)
print(f"🔗 NGROK URL: {public_url}")
print("=" * 60)
print()
print(f"Στο n8n, βάλε σε κάθε Generate node:")
print(f"  {public_url}/api/chat")
print()
print(f"Health check:")
print(f"  {public_url}/health")
print("=" * 60)

# Start FastAPI in a background thread (Colab has its own event loop)
thread = Thread(target=uvicorn.run, args=(app,),
                kwargs={"host": "0.0.0.0", "port": 8000, "log_level": "info"},
                daemon=True)
thread.start()
print("\n✅ Server running! Κράτα αυτό το tab ανοιχτό.")

In [ ]:
# ── (Optional) Test the API locally before connecting n8n ──
# Run this in a separate cell BEFORE starting the server above,
# or open a new notebook tab to test.

# import requests
# resp = requests.post("http://localhost:8000/api/chat", json={
#     "model": "krikri-8b",
#     "messages": [
#         {"role": "system", "content": "Είσαι ο Γιώργος Τροχίδης."},
#         {"role": "user", "content": "Πού σπούδασες;"},
#     ],
#     "stream": False,
#     "options": {"temperature": 0.5, "num_predict": 100}
# })
# print(resp.json())